# 9.6 — Checkpoint Recovery

**Chapter 9, §9.7.2, and Exercise 5.** *What does a streaming query forget when it
is restarted, and what does a checkpoint make it remember?*

A batch job that dies can be rerun from its input. A streaming query cannot: it has
been consuming an input that keeps moving, and starting again from the beginning is
either wrong or ruinously expensive. The checkpoint is what closes that gap. It
records which input the query has already consumed and what state it had
accumulated, so a restart resumes instead of repeating.

This notebook runs the same query four times over the same three files: twice with
no checkpoint, twice with one. The difference is how much the second run of each
pair delivers to its sink, and the checkpoint itself turns out to be a small
directory of ordinary files you can open and read.

In [1]:
import json, os, shutil, tempfile, time

from pyspark.sql import SparkSession

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch09-checkpoint")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

spark = (
    SparkSession.builder.appName("CS777-9.6")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

## Three files, which are the whole of the stream

Each holds two readings. Nothing here aggregates: the query filters and projects.
Keeping it stateless means the only thing a restart can get wrong is which files
it has already consumed, which is precisely what is being measured.

In [2]:
INPUT = os.path.join(SCRATCH, "input")
os.makedirs(INPUT, exist_ok=True)

FILES = [
    [("boston", 12.1), ("denver", 3.8)],
    [("miami", 27.0), ("seattle", 9.4)],
    [("boston", 12.6), ("denver", 4.1)],
]
for order, rows in enumerate(FILES):
    path = os.path.join(INPUT, f"{order}.csv")
    with open(path, "w") as fh:
        for station, temperature in rows:
            fh.write(f"{station},{temperature}\n")
    stamp = time.time() - 3600 + 10 * order
    os.utime(path, (stamp, stamp))

print(f"{len(FILES)} files, {sum(len(f) for f in FILES)} rows in total")

3 files, 6 rows in total


In [3]:
SCHEMA = "station STRING, temperature_c DOUBLE"

# The sink is a callback that simply records what it was handed. foreachBatch is
# used rather than one of the built-in sinks because it has no protection of its
# own: whatever Spark delivers, it accepts, which is what makes the replay
# visible instead of quietly absorbed.
def run(checkpoint=None, label=""):
    delivered = []

    def sink(frame, batch_id):
        delivered.append((batch_id, frame.count()))

    readings = (
        spark.readStream.schema(SCHEMA)
        .option("maxFilesPerTrigger", 1)
        .csv(INPUT)
        .where("temperature_c > 0")
    )
    writer = readings.writeStream.foreachBatch(sink).trigger(availableNow=True)
    if checkpoint:
        writer = writer.option("checkpointLocation", checkpoint)
    # With no checkpoint directory given, Spark makes a temporary one and
    # discards it when the query ends, which is exactly the situation below.
    query = writer.start()
    query.awaitTermination()
    total = sum(rows for _, rows in delivered)
    print(f"{label}: {len(delivered)} batches, {total} rows delivered to the sink")
    return total

## Without a checkpoint, a restart is a replay

The second run has no memory of the first. It finds three files it has never heard
of and processes all three again. The sink receives every reading a second time.

In [4]:
first = run(label="run 1, no checkpoint")
second = run(label="run 2, no checkpoint")
assert first == second == 6
print("everything was processed a second time")

run 1, no checkpoint: 3 batches, 6 rows delivered to the sink


run 2, no checkpoint: 3 batches, 6 rows delivered to the sink
everything was processed a second time


## With a checkpoint, a restart resumes

Same query, same input, but progress is now recorded somewhere that outlives the
query. The second run finds nothing it has not already processed and delivers
nothing.

In [5]:
CHECKPOINT = os.path.join(SCRATCH, "checkpoint")

first = run(checkpoint=CHECKPOINT, label="run 1, checkpointed")
second = run(checkpoint=CHECKPOINT, label="run 2, checkpointed")
assert first == 6 and second == 0
print("nothing was reprocessed")

run 1, checkpointed: 3 batches, 6 rows delivered to the sink
run 2, checkpointed: 0 batches, 0 rows delivered to the sink
nothing was reprocessed


## What is actually in a checkpoint

Three things, and they are ordinary files. The **offset log** records the input
each micro-batch was planned to consume, the **commit log** records which of those
batches finished, and the state store holds whatever a stateful operator had
accumulated. On restart Spark compares the first two to find the last batch that
completed, restores the state, and carries on from there.

In [6]:
for root, _, names in sorted(os.walk(CHECKPOINT)):
    for name in sorted(names):
        if not name.startswith("."):
            print(os.path.relpath(os.path.join(root, name), CHECKPOINT))

metadata
commits/0
commits/1
commits/2
offsets/0
offsets/1
offsets/2
sources/0/0
sources/0/1
sources/0/2


In [7]:
# The offset log says how far into the source each batch was planned to go. For
# a file source that position is a number, and the file names behind it live in
# the source's own log alongside it.
print("--- offsets/0 ---")
print(open(os.path.join(CHECKPOINT, "offsets", "0")).read().strip().split("\n")[-1])

print("\n--- commits/0 ---   (this batch finished)")
print(open(os.path.join(CHECKPOINT, "commits", "0")).read().strip()[:120])

print("\n--- sources/0/0 ---   (the files batch 0 consumed)")
for line in open(os.path.join(CHECKPOINT, "sources", "0", "0")).read().strip().split("\n"):
    print(json.loads(line)["path"].rsplit("/", 1)[-1] if line.startswith("{") else line)

--- offsets/0 ---
{"logOffset":0}

--- commits/0 ---   (this batch finished)
v1
{"nextBatchWatermarkMs":0}

--- sources/0/0 ---   (the files batch 0 consumed)
v1
0.csv


## What this shows

The checkpoint is not an optimization and it is not the RDD checkpointing of the
performance chapter, which shortens a lineage and is a tuning decision. This one
records position and state so that a restart is a resumption, and for any
long-running query it is not optional.

One sink is an exception worth knowing about. Spark's built-in file sink keeps its
own metadata log beside the output, so it will not write the same batch twice even
when the query has no durable checkpoint; that log is why the chapter lists the
file sink as exactly-once. A `foreachBatch` sink has no such protection, which is
what notebook 9.8 is about.

Exercise 5 asks you to see this on the real pipeline. `09.04 GDELT Stream.py`
ships with its checkpoint deliberately switched off, so stopping and starting it
reprocesses every export in the watched directory. Run it, restart it, watch the
counts start over, and then pass `--checkpoint` and watch them not. In production
the checkpoint belongs on durable storage that outlives the machine, which in
practice means object storage rather than a local disk.

In [8]:
spark.stop()